In [180]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error
import time
#import pygwalker as pyg
from datetime import datetime
import os
#from ydata_profiling import ProfileReport
import csv
import seaborn as sns

In [246]:
# train_df = pd.read_csv('../artifacts/train.csv')
# test_df = pd.read_csv('../artifacts/test.csv')
df = pd.read_csv('../artifacts/raw.csv')

C:\Users\Dhvanish\AppData\Local\Temp\ipykernel_23464\13393110.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../artifacts/raw.csv')


## Imputing missing values ##

In [247]:
df['CompetitionDistance'].fillna(0,inplace=True)
#train_df.isnull().sum()

In [248]:
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

df['CompetitionOpen_missing'] = df['CompetitionOpenSinceYear'].isna().astype(int)
df['CompetitionOpenSinceMonth'].fillna(1,inplace=True)
df['CompetitionOpenSinceYear'].fillna(df['Year'].min(),inplace=True)
df['CompetitionOpen'] = 12 * (df['Year'] - df['CompetitionOpenSinceYear']) + (df['Month'] - 
                                                                              df['CompetitionOpenSinceMonth'])
df['CompetitionOpen'] = df['CompetitionOpen'].apply(lambda x: max(x, 0))


In [249]:
df['WeekOfYear'] = df['Date'].dt.isocalendar().week
df['Promo2OpenSinceMonths'] = 12 * (df['Year'] - df['Promo2SinceYear']) + (df['WeekOfYear'] - 
                                                                           df['Promo2SinceWeek']) / 4.0
df['Promo2OpenSinceMonths'] = df['Promo2OpenSinceMonths'].apply(lambda x: max(x, 0) if pd.notnull(x) else 0)
df.loc[df['Promo2'] == 0, 'Promo2OpenSinceMonths'] = 0
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df['MonthStr'] = df['Month'].map(month_map)
promo_months = df['PromoInterval'].fillna('').str.split(',')
df['IsPromoMonth'] = [
    1 if m in months else 0 
    for m, months in zip(df['MonthStr'], promo_months)
]


## Feature Engineering ##

In [250]:
df['StateHoliday'] = np.where((df['StateHoliday'] == '0') | (df['StateHoliday'] == 0),0,1)

In [251]:
df['Assortment'] = np.where(df['Assortment'] == 'b','b','other')

In [253]:
df['12_month'].value_counts()

12_month
0    953659
1     63550
Name: count, dtype: int64

In [254]:
df.sample(5)

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,Year,Month,Day,CompetitionOpen_missing,CompetitionOpen,WeekOfYear,Promo2OpenSinceMonths,MonthStr,IsPromoMonth,12_month
463622,568,1,2014-05-12,3913,370,1,0,0,0,d,...,2014,5,12,1,16.0,20,16.75,May,0,0
675061,157,7,2013-11-03,0,0,0,0,0,0,a,...,2013,11,3,0,109.0,44,0.00,Nov,0,0
450828,39,5,2014-05-23,4582,597,1,1,0,0,a,...,2014,5,23,0,91.0,21,9.50,May,1,0
82415,1021,2,2015-05-19,9894,1127,1,1,0,0,a,...,2015,5,19,0,48.0,21,0.00,May,0,0
626631,787,2,2013-12-17,16905,1616,1,1,0,0,c,...,2013,12,17,0,54.0,51,0.00,Dec,0,1


In [255]:
df = df[df['Open'] == 1].copy()

In [256]:
df.shape

(844392, 28)

In [257]:
df = df.sort_values('Date')

cutoff_date = '2015-06-01'

train = df[df['Date'] < cutoff_date]
valid = df[df['Date'] >= cutoff_date]

store_avg_sales = train.groupby('Store')['Sales'].mean().rename('Store_avg_sales')
store_avg_customers = train.groupby('Store')['Customers'].mean().rename('Store_avg_customers')
train = train.merge(store_avg_sales, on='Store', how='left')
train = train.merge(store_avg_customers, on='Store', how='left')
valid = valid.merge(store_avg_sales, on='Store', how='left')
valid = valid.merge(store_avg_customers, on='Store', how='left')

X_train = train.drop(['Sales', 'Date'], axis=1)
y_train = train['Sales']

X_test = valid.drop(['Sales', 'Date'], axis=1)
y_test = valid['Sales']

In [258]:
X_train.drop(['Customers','CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear','PromoInterval','MonthStr'], axis=1, inplace=True)
X_test.drop(['Customers','CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear','PromoInterval','MonthStr'], axis=1, inplace=True)

In [259]:
num_col=['CompetitionDistance','CompetitionOpen','Promo2OpenSinceMonths','Store_avg_sales','Store_avg_customers']
cat_col = ['StoreType','Assortment','Year']

In [260]:
preprocessor = ColumnTransformer([
    ('scl',StandardScaler(),num_col),
    ('ohe',OneHotEncoder(drop='first'),cat_col)    
],remainder='passthrough')

In [261]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [262]:
y_train_log = np.log1p(y_train)

In [263]:
results_file = 'rmsep_score.csv'
file_exists = os.path.isfile(results_file)

models = {
    "XGBRegressor" : XGBRegressor(tree_method='hist',n_jobs=-1),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=50,max_depth=15,n_jobs=-1),
    'LinearRegression': LinearRegression(),
    'LGBMRegressor': LGBMRegressor(n_jobs=-1)
}

results = []

for model_name,model in models.items():
    y_test_mean = np.mean(y_test)

    training_start = time.perf_counter()
    model.fit(X_train,y_train_log)
    training_stop = time.perf_counter()
    training_time_taken = training_stop - training_start

    prediction_start = time.perf_counter()
    pred_log = model.predict(X_test)
    prediction_stop = time.perf_counter()

    Prediction = np.expm1(pred_log)
    #prediction = pred_log

    prediction_time_taken = prediction_stop-prediction_start
    # rmse = root_mean_squared_error(y_test,prediction)
    # rmsep = rmse/y_test_mean

    def rmspe(y_true, y_pred):
        mask = y_true != 0
        return np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2))

    score = rmspe(y_test.values, Prediction)


    print(f'{model_name}: {score*100:.2f}%\n')
    print(f'Training time taken for {model_name}: {training_time_taken:.4f}\n')
    print(f'prediction time taken for {model_name}: {prediction_time_taken:.4f}\n')

    results.append({
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'model_name': model_name,
        #'rmse': rmse,
        'rmsep_percent': score * 100,
        'prediction_time_taken_sec': prediction_time_taken,
        'training_time_taken_sec': training_time_taken
    })

with open(results_file, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['timestamp', 'model_name', 'rmsep_percent', 'prediction_time_taken_sec','training_time_taken_sec'])
    if not file_exists:
        writer.writeheader()
    writer.writerows(results)

XGBRegressor: 16.28%

Training time taken for XGBRegressor: 7.6745

prediction time taken for XGBRegressor: 0.1168

RandomForestRegressor: 17.26%

Training time taken for RandomForestRegressor: 92.9594

prediction time taken for RandomForestRegressor: 0.1529

LinearRegression: 23.66%

Training time taken for LinearRegression: 2.1153

prediction time taken for LinearRegression: 0.0686

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.050006 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1662
[LightGBM] [Info] Number of data points in the train set: 785781, number of used features: 23
[LightGBM] [Info] Start training from score 8.754656
LGBMRegressor: 18.36%

Training time taken for LGBMRegressor: 5.7242

prediction time taken for LGBMRegressor: 0.1525

